In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from torchvision.datasets import ImageFolder
from torchvision import transforms
import os
import kagglehub
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

train_dir = os.path.join(path, "PlantVillage", "train")
test_dir = os.path.join(path, "PlantVillage", "test")

train_dataset = ImageFolder(root=train_dir, transform=transform)
test_dataset  = ImageFolder(root=test_dir,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)  #batch size 32 means 32 pictures in each batch
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2) #shuffling doesnt matter cuz we are just validating


In [ ]:
import random
import numpy as np

# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8, title="Dataset Samples"):
    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        class_name = dataset.classes[label]
        # Display image
        axes[i].imshow(image)
        axes[i].set_title(f"{class_name}\n(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
visualize_samples(train_dataset, num_samples=8, title="Training Dataset Samples")

In [ ]:
from torch import nn
class CustomCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1), #[B,8,32,32]
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d(8, 16, kernel_size=3, padding=1), #[B,16,32,32]
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1), #[B,32,32,32]
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, padding=1), #[B,64,32,32]
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1), #[B,64,32,32]
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*32*32, 64),
            nn.ReLU(),
            #nn.Dropout(0.3),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
# Write your code here
from tqdm import tqdm
def accuracy_from_logits(logits, labels):
    # TO-DO: Get predicted class indices
    # HINT: Use torch.argmax with dim=1
    preds = torch.argmax(logits, dim=1)

    # TO-DO: Calculate and return accuracy
    # HINT: Compare preds with labels, convert to float, mean, then .item()
    return (preds == labels).to(dtype=float).mean().item()

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(model(images), labels)
        loss.backward()   #calculating the gradient
        optimizer.step()  #updates the weights
        total_loss += loss.item()
        total_acc += accuracy_from_logits(outputs.detach(), labels)


    return total_loss / len(loader), total_acc / len(loader)


def validate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            #correct += (outputs.argmax(dim=1) == labels).sum().item()   #without .item() it returns a tensor containg a single value so we use .item() to get the raw number integer or float
            #total += labels.size(0)
            total_acc += accuracy_from_logits(outputs.detach(), labels)
            loss = criterion(model(images), labels)
            total_loss += loss.item()

    return total_loss / len(loader), total_acc / len(loader)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNN().to(device)
model = model.to(device)
model

In [ ]:
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

num_epochs = 10
history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    test_loss, test_acc = validate(model, test_loader, criterion)

    # Store history
    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)


    print(f'Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}')

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), history["train_loss"] , label="train_loss", marker='o')
plt.plot(range(1, num_epochs+1), history["test_loss"], label= "test_loss" , marker='o')
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), history["train_acc"], label="train_acc", marker='o')
plt.plot(range(1, num_epochs+1), history["test_acc"] , label="test_acc", marker='o')
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend()

plt.show()##


In [ ]:
# Write your code here
from torch import nn
class CustomCNN_Res(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features1 = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1), #[B,8,32,32]
            nn.BatchNorm2d(8),
            nn.ReLU()
        )

        #2nd conv layer
        self.enc2 = nn.Conv2d(8, 16, kernel_size=3, padding=1), #[B,16,32,32]

        self.features2 = nn.Sequential(
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=3, padding=1), #[B,32,32,32]
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.enc4 = nn.Conv2d(32, 64, kernel_size=3, padding=1), #[B,64,32,32]

        self.features3 = nn.Sequential(
            #4th Conv Layer
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1), #[B,64,32,32]
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*32*32, 64),
            nn.ReLU(),
            #nn.Dropout(0.3),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        x = self.features1(x)
        x = self.enc2(x)
        enc2_val = x
        x = self.features2(x)
        x = torch.cat([x,enc2_val], dim =1) #residual connection
        #no need for pooling because the dims remained same due to padding
        x = self.enc4(x)
        x = self.features3(x)
        x = self.classifier(x)
        return x